# AML 3303 — Week 3 In-Class Lab (30 minutes)
### pandas + NumPy only

## Scenario

**NorthLine Electronics** is an online store. The ops team exported two tables for you:
`customers` and `orders`. They want a quick read on which product categories actually
make money, and which city orders the most.

The export is realistic, which means it is slightly messy: a missing quantity, a missing
price, a duplicate order, and one order from a customer who is not in the customer list.

Fill in every cell marked `# TODO`. The hints sit directly above each one.

## 0. Setup

Run this cell once. Do not change it.

In [ ]:
# Run this cell first. It creates the two DataFrames you will work with.
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "name": ["Aarav Shah","Bianca Rossi","Chen Wei","Diya Patel",
             "Emeka Obi","Farah Haddad","Grace Kim","Hugo Martin"],
    "city": ["Ottawa","Toronto","Ottawa","Montreal",
             "Toronto","Calgary","Montreal","Ottawa"],
})

orders = pd.DataFrame({
    "order_id":   [101,102,103,104,105,106,107,108,109,110,
                   111,112,113,114,115,116,117,118,119,120,118,121],
    "customer_id":[  1,  2,  1,  3,  4,  5,  2,  6,  3,  7,
                     8,  5,  1,  4,  6,  7,  2,  8,  3,  5,  8,  9],
    "order_date": ["2024-05-02","2024-05-02","2024-05-04","2024-05-05","2024-05-07",
                   "2024-05-08","2024-05-09","2024-05-11","2024-05-12","2024-05-13",
                   "2024-05-15","2024-05-16","2024-05-18","2024-05-19","2024-05-20",
                   "2024-05-22","2024-05-23","2024-05-25","2024-05-26","2024-05-28",
                   "2024-05-25","2024-05-29"],
    "category":   ["Laptops","Headphones","Headphones","Monitors","Laptops",
                   "Keyboards","Monitors","Laptops","Keyboards","Headphones",
                   "Monitors","Laptops","Keyboards","Headphones","Monitors",
                   "Laptops","Keyboards","Headphones","Laptops","Monitors",
                   "Headphones","Keyboards"],
    "quantity":   [1,2,1,2,1,3,1,1,np.nan,1,4,1,2,1,1,1,1,2,1,2,2,1],
    "unit_price": [1299.99,89.50,89.50,219.00,1499.00,45.00,219.00,1099.00,45.00,np.nan,
                   219.00,1299.99,45.00,89.50,219.00,1499.00,45.00,89.50,1299.99,219.00,
                   89.50,45.00],
    "status":     ["delivered","delivered","returned","delivered","cancelled",
                   "delivered","delivered","delivered","delivered","delivered",
                   "delivered","returned","delivered","delivered","delivered",
                   "delivered","cancelled","delivered","delivered","delivered",
                   "delivered","delivered"],
})

print("orders:", orders.shape, " customers:", customers.shape)
orders.head()

## 1. First look (3 min)

**Hints**
- `df.shape` → rows and columns
- `df.info()` → column types and non-null counts
- `df.describe()` → quick numeric summary
- `df["category"].value_counts()` → how many orders per category

In [ ]:
# Size of the table (rows, columns)
print(orders.shape)

# Column types + non-null counts (spot missing values)
orders.info()

# Number of orders per category
orders["category"].value_counts()

## 2. A first (wrong) answer (5 min)

Compute revenue per category **before** cleaning anything, counting delivered orders only.
Keep this result: you will compare against it at the end.

**Hints**
- Filter rows: `orders[orders["status"] == "delivered"]`
- New column: `df["revenue"] = df["quantity"] * df["unit_price"]`
- Group: `df.groupby("category")["revenue"].sum().sort_values(ascending=False)`

In [ ]:
# Keep only delivered orders
delivered_raw = orders[orders["status"] == "delivered"].copy()

# Revenue = quantity x price (NaNs give NaN, which sum() skips)
delivered_raw["revenue"] = delivered_raw["quantity"] * delivered_raw["unit_price"]

# Total revenue per category, biggest first
revenue_raw = delivered_raw.groupby("category")["revenue"].sum().sort_values(ascending=False)
revenue_raw

## 3. Find the problems (5 min)

**Hints**
- Duplicates: `df.duplicated().sum()`, and `df[df.duplicated(keep=False)]` to see them
- Missing values: `df.isna().sum()`
- Orphan rows: `~orders["customer_id"].isin(customers["customer_id"])`

In [ ]:
# 1. Count exact duplicate rows, then show every copy
print("Duplicates:", orders.duplicated().sum())
display(orders[orders.duplicated(keep=False)])

# 2. Missing values per column
print(orders.isna().sum())

# 3. Orders whose customer_id is not in the customers table
orphans = orders[~orders["customer_id"].isin(customers["customer_id"])]
orphans

## 4. Clean it (7 min)

**Hints**
- `orders_clean = orders.drop_duplicates()`
- Fill a price with the median for that category:
  `df.groupby("category")["unit_price"].transform("median")`
- `df["quantity"].fillna(1)` — one unit is a reasonable default for this shop
- `pd.to_datetime(df["order_date"])`
- Make quantity an integer at the end: `.astype(int)`

In [ ]:
orders_clean = orders.copy()

# 1. Remove the duplicate order (order 118)
orders_clean = orders_clean.drop_duplicates()

# 2. Missing price -> median price of the same category
cat_median = orders_clean.groupby("category")["unit_price"].transform("median")
orders_clean["unit_price"] = orders_clean["unit_price"].fillna(cat_median)

# 3. Missing quantity -> 1 unit, then make it a whole number
orders_clean["quantity"] = orders_clean["quantity"].fillna(1).astype(int)

# 4. Text dates -> real datetime
orders_clean["order_date"] = pd.to_datetime(orders_clean["order_date"])

orders_clean.info()

## 5. NumPy: revenue column (3 min)

Cancelled and returned orders earn **no** revenue.

**Hints**
- `np.where(condition, value_if_true, value_if_false)`
- Condition: `orders_clean["status"] == "delivered"`
- `np.round(x, 2)` if you want tidy numbers

In [ ]:
# Revenue only for delivered orders; cancelled/returned = 0
orders_clean["revenue"] = np.where(
    orders_clean["status"] == "delivered",
    np.round(orders_clean["quantity"] * orders_clean["unit_price"], 2),
    0,
)

# Quick revenue stats
print("Total revenue:     ", round(orders_clean["revenue"].sum(), 2))
print("Mean order value:  ", round(orders_clean["revenue"].mean(), 2))
print("Largest order:     ", orders_clean["revenue"].max())

## 6. Merge and answer the business question (5 min)

**Hints**
- `orders_clean.merge(customers, on="customer_id", how="left")`
- A left merge keeps the order from the unknown customer, with `city` as NaN. That is
  usually what you want: do not silently delete money.
- `.groupby("city")["revenue"].sum().sort_values(ascending=False)`

In [ ]:
# Left merge keeps every order (unknown customer gets city = NaN)
merged = orders_clean.merge(customers, on="customer_id", how="left")

# Revenue per city; dropna=False keeps the unknown-city money visible
revenue_by_city = merged.groupby("city", dropna=False)["revenue"].sum().sort_values(ascending=False)
revenue_by_city

## 7. Compare with step 2 (2 min)

**Hint:** `pd.DataFrame({"raw": revenue_raw, "clean": revenue_clean})` lines the two up side by side.

In [ ]:
# Revenue per category after cleaning
revenue_clean = orders_clean.groupby("category")["revenue"].sum().sort_values(ascending=False)

# Side-by-side view plus the difference
comparison = pd.DataFrame({"raw": revenue_raw, "clean": revenue_clean})
comparison["diff"] = comparison["clean"] - comparison["raw"]
comparison

**Question:** two categories changed. Which ones, and what caused each change?

*Your answer:*
- **Headphones** dropped by 89.50 (626.50 → 537.00). Removing the duplicate order 118 took off 179.00. Filling order 110's missing price with the category median (89.50) added 89.50 back.
- **Keyboards** rose by 45.00 (270.00 → 315.00). Order 109 had no quantity, so the raw calculation gave NaN and `sum()` skipped it. Filling it with 1 added 1 × 45.00.


## 8. Wrap up (optional, if time)

Write two sentences for the NorthLine ops team: which category to push, and one data
quality issue they should fix at the source.

*Your answer:*
Push **Laptops**: they bring in about 5,200 of the 8,240 in delivered revenue, more than every other category combined. At the source, stop duplicate order exports (order 118 was exported twice), and make quantity and price required fields so orders can't be saved without them.

---

### Submission
Save the notebook, push it to your GitHub repo, and paste the repo link in the class channel.

### Stretch goals
1. Which customer spent the most? (merge, then group by name)
2. Add a `week` column from `order_date` and show revenue per week.
   Hint: `df["order_date"].dt.isocalendar().week`
3. Ask Claude or Copilot to write the cleaning step for you, then compare it with yours.
   Did it make the same choices about the missing quantity? Was it right?
